# 19. Which tier was the customer in when they ordered?

**Track:** sql | **Difficulty:** medium | **Tags:** scd2, slowly changing dimension, point-in-time join, range join, between

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("019-scd2-point-in-time-lookup")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "customer_dim": """
UEFSMRUEFcADFfoBTBU4FQASAADgAQjXAgAFAQDMDQgA0A0IAMkNCADKDQgAzg0IAMMNCADIDQgA0w0IAMUNCADRDQgAzQ0IAMYN
CADBDQgAvw0IANUNCAC9DQgAxw0IANINCADCDQgAvg0IANgNCADPDQgAyw0IANQNCADEDQg81gIAAAAAAADAAgAAAAAAABUAFWwV
cCwVigEVEBUGFQYcGAjYAgAAAAAAABgIvQIAAAAAAAAWACgI2AIAAAAAAAAYCL0CAAAAAAAAEREAAAA21AMAAACKAQEFEyCIQYo5
KB2FwGJlhWBCc6k9GI2ScdJqLySq4az3gFcdkAg4daK9EpNFT40AABUEFTwVQEwVBhUAEgAAHnQEAAAAR09MRAYAAABTSUxWRVII
AAAAUExBVElOVU0VABU2FTosFYoBFRAVBhUGHDYAKAZTSUxWRVIYBEdPTEQREQAAABtoAwAAAIoBAQITkBZhUJKBFBpQVBJCilKg
pWQBFQQV2AMV4ANMFXYVABIAAOwB8Ov0TQAAp04AAFlNAABsTQAAOU4AAIxNAABrTQAAxU0AACtOAAANTgAARk0AAOJNAABJTQAA
Yk0AAH9NAAAsTgAAt00AADJOAADOTQAAQE4AAG1NAADxTQAAqE0AAEVNAABkTQAA4U0AANlNAADYTQAAhk0AALFNAAB3TQAAo00A
AF1NAAAnTQAA2k0AAHRNAABDTQAAAU4AAJZNAAAvTQAAk00AAJxNAAAgTQAAE00AAKlNAADbTQAAoE0AAFNNAAAUTQAAI00AAHtN
AADtTQAAD04AAFhOAAAdTQAAQU0AAPtNAACOTQAAOk4AABUAFX4VhAEsFYoBFRAVBhUGHBgEp04AABgEE00AABYAKASnTgAAGAQT
TQAAEREAAAA/8D4DAAAAigEBBhNAIAxEYRxIUigL0zgPFEkTVSmDdWGZtnFD531gGIqjR16leaKpurKt67xwLM90bd94nusWAAAV
BBW4AhXAAkwVThUAEgAAnAHwm/tNAADZTQAAWE4AAMVNAAB0TQAAK04AAH9NAAC3TQAAD04AAKdOAAAyTgAAzk0AANtNAADxTQAA
k00AAEBOAADhTQAAnE0AAGRNAAA5TgAA2E0AAKhNAAA6TgAA7U0AAGxNAABGTQAAU00AAPRNAAANTgAA2k0AAKlNAACMTQAAR00A
AI5NAACGTQAALE4AAAFOAACxTQAA4k0AABUAFWgVbCwVigEVEBUGFQYcGASnTgAAGARGTQAAFjgoBKdOAAAYBEZNAAAREQAAADTM
CgAAABNgfi4r4XP+xxcGDUAgDERhHEiiLEzjPFAkTVRlXVimbVznfWBYGeJIliYAAAAAABUEGVw1ABgGc2NoZW1hFQgAFQQlAhgL
Y3VzdG9tZXJfaWQAFQwlAhgEdGllciUATBwAAAAVAiUCGAp2YWxpZF9mcm9tJQxMbAAAABUCJQIYCHZhbGlkX3RvJQxMbAAAABaK
ARkcGUwmABwVBBk1AAYQGRgLY3VzdG9tZXJfaWQVAhaKARbMBRaKBCaiAiYIHBgI2AIAAAAAAAAYCL0CAAAAAAAAFgAoCNgCAAAA
AAAAGAi9AgAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCKAQAAACYAHBUMGTUABhAZGAR0aWVyFQIWigEW2gEW4gEm7gQm
kgQcNgAoBlNJTFZFUhgER09MRBERABksFQQVABUCABUAFRAVAgA8FqgGGQYZJgCKAQAAACYAHBUCGTUABhAZGAp2YWxpZF9mcm9t
FQIWigEW2AUW5gUm9Akm9AUcGASnTgAAGAQTTQAAFgAoBKdOAAAYBBNNAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAigEAAAAm
ABwVAhk1AAYQGRgIdmFsaWRfdG8VAhaKARagBBasBCa6DibaCxwYBKdOAAAYBEZNAAAWOCgEp04AABgERk0AABERABksFQQVABUC
ABUAFRAVAgA8KQYZJjhSAAAAFp4RFooBJggW/g8AGRwYDEFSUk9XOnNjaGVtYRiMAy8vLy8veUFCQUFBUUFBQUFBQUFLQUF3QUJn
QUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFRQUFBQzBBQUFBZEFBQUFEZ0FBQUFFQUFBQWJQLy8v
d0FBQVFnUUFBQUFIQUFBQUFRQUFBQUFBQUFBQ0FBQUFIWmhiR2xrWDNSdkFBQUFBTTcvLy84QUFBQUFuUC8vL3dBQUFRZ1FBQUFB
SkFBQUFBUUFBQUFBQUFBQUNnQUFBSFpoYkdsa1gyWnliMjBBQUFBQUJnQUlBQVlBQmdBQUFBQUFBQURVLy8vL0FBQUJCUkFBQUFB
Y0FBQUFCQUFBQUFBQUFBQUVBQUFBZEdsbGNnQUFBQUFFQUFRQUJBQUFBQkFBRkFBSUFBWUFCd0FNQUFBQUVBQVFBQUFBQUFBQkFo
QUFBQUFrQUFBQUJBQUFBQUFBQUFBTEFBQUFZM1Z6ZEc5dFpYSmZhV1FBQ0FBTUFBZ0FCd0FJQUFBQUFBQUFBVUFBQUFBPQAYIHBh
cnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGUwcAAAcAAAcAAAcAAAAyAMAAFBBUjE=
""",
    "orders": """
UEFSMRUEFfAYFdYMTBWOAxUAEgAAuAwMgTgBAAEBAIINCACDDQgAhA0IAIUNCACGDQgAhw0IAIgNCACJDQgAig0IAIsNCACMDQgA
jQ0IAI4NCACPDQgAkA0IAJENCACSDQgAkw0IAJQNCACVDQgAlg0IAJcNCACYDQgAmQ0IAJoNCACbDQgAnA0IAJ0NCACeDQgAnw0I
AKANCAChDQgAog0IAKMNCACkDQgApQ0IAKYNCACnDQgAqA0IAKkNCACqDQgAqw0IAKwNCACtDQgArg0IAK8NCACwDQgAsQ0IALIN
CACzDQgAtA0IALUNCAC2DQgAtw0IALgNCAC5DQgAug0IALsNCAC8DQgAvQ0IAL4NCAC/DQgAwA0IAMENCADCDQgAww0IAMQNCADF
DQgAxg0IAMcNCADIDQgAyQ0IAMoNCADLDQgAzA0IAM0NCADODQgAzw0IANANCADRDQgA0g0IANMNCADUDQgA1Q0IANYNCADXDQgA
2A0IANkNCADaDQgA2w0IANwNCADdDQgA3g0IAN8NCADgDQgA4Q0IAOINCADjDQgA5A0IAOUNCADmDQgA5w0IAOgNCADpDQgA6g0I
AOsNCADsDQgA7Q0IAO4NCADvDQgA8A0IAPENCADyDQgA8w0IAPQNCAD1DQgA9g0IAPcNCAD4DQgA+Q0IAPoNCAD7DQgA/A0IAP0N
CAD+DQgA/w0IBAA5afgAAQ0IAAINCAADDQgABA0IAAUNCAAGDQgABw0IAAgNCAAJDQgACg0IAAsNCAAMDQgADQ0IAA4NCAAPDQgA
EA0IABENCAASDQgAEw0IABQNCAAVDQgAFg0IABcNCAAYDQgAGQ0IABoNCAAbDQgAHA0IAB0NCAAeDQgAHw0IACANCAAhDQgAIg0I
ACMNCAAkDQgAJQ0IACYNCAAnDQgAKA0IACkNCAAqDQgAKw0IACwNCAAtDQgALg0IAC8NCAAwDQgAMQ0IADINCAAzDQgANA0IADUN
CAA2DQgANw0IADgNCAA5DQgAOg0IADsNCAA8DQgAPQ0IAD4NCAA/DQgAQA0IAEENCABCDQgAQw0IAEQNCABFDQg8RjkBAAAAAABH
OQEAAAAAABUAFaIDFaoDLBWOAxUQFQYVBhwYCEc5AQAAAAAAGAiBOAEAAAAAABYAKAhHOQEAAAAAABgIgTgBAAAAAAAREQAAANEB
8NADAAAAjgMBCDMAAQIDBAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJSYnKCkqKywtLi8wMTIzNDU2Nzg5Ojs8PT4/
QEFCQ0RFRkdISUpLTE1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXp7fH1+f4CBgoOEhYaHiImK
i4yNjo+QkZKTlJWWl5iZmpucnZ6foKGio6SlpqeoqaqrrK2ur7CxsrO0tba3uLm6u7y9vr/AwcLDxMXGABUEFdADFYYCTBU6FQAS
AADoAQjYAgAFAQDODQgAyw0IANINCADBDQgA1g0IAM8NCADTDQgAww0IAMoNCAC/DQgAwA0IAMkNCADCDQgAyA0IANENCADNDQgE
HwMFhwQA1w0QAL4NCADHDQgA0A0IAMQNCADGDQgAxQ0IANQNCADVDQg8vQIAAAAAAADMAgAAAAAAABUAFYwCFZQCLBWOAxUQFQYV
BhwYCB8DAAAAAAAAGAi9AgAAAAAAABYAKAgfAwAAAAAAABgIvQIAAAAAAAAREQAAAIYB8IUDAAAAjgMBBTMgiEGKGQcltVhrxT0G
oyxMLDnTm9RUey0bEw2qEpY4D6QRfGtT3MpEzq0dxtGhOTjEIExMB+6pNxHQYGpnP3oGEirriZd6SWFF8mU0ipcxlc9WsMdtmPIw
UXZPG802npbfyiZGJd1CMk537xJETY3IsFWaIIwmwRBXpKmCBRUEFeAJFeoJTBW4AhUAEgAA8AT0bwKjTQAAvE0AAJlNAAArTgAA
hU0AAOdNAAAxTQAAc00AAD9NAACyTQAAEU0AAFxOAADZTQAAGE0AAI5NAADHTQAAFU4AAE5NAAAOTgAAKk0AAHhNAAA9TgAAX00A
AP1NAABsTQAA+U0AAIpNAACMTQAA300AAMVNAABWTQAAzk0AADtOAABGTgAAWU0AACJNAACmTQAAWE0AAOJNAAB6TQAAZE4AAKRN
AABfTgAAPk0AAHVOAABETgAAT00AAGFOAAAPTQAAfE0AALVNAABaTQAAYE4AAANOAADeTQAAe00AACxOAAAgTgAAkE0AAIhNAAAz
TgAA7U0AACZNAABuTQAAQU0AAOFNAAA0TQAAJU0AAP9NAAASTgAAbk4AAAxNAAA9TQAAXU4AAEdNAABzTgAATE0AAF5NAAArTQAA
B04AAJNNAACrTQAA1k0AACpOAABBTgAA100AAAFOAABbTQAAH04AAGVNAABDTgAAKE0AAPFNAABeTgAAV04AAL5NAABgTQAARk0A
AFFOAAASTQAAcE0AAG9OAAB4TgAACE4AAJZNAAB0TQAAWE4AAOVNAAA1TQAAP04AAB9NAAD7TQAA800AAClNAAAkTQAA9U0AAM1N
AACSTQAAQE4AAGVOAACoTQAA2E0AAIlNAAAuTQAAfU0AACJOAAAaTQAADU4AAL9NAAAVTQAAmE0AAMJNAAAmTgAAak4AAPRNAAA6
TgAA2k0AAFBNAABpTgAAPE4AAIRNAAA6TQAANU4AALRNAABSTQAA+E0AADFOAADvTQAAUE4AAKJNAAC3TQAAl00AAGNOAACcTQAA
KU4AAEJNAAAVABWiAxWqAywVjgMVEBUGFQYcGAR4TgAAGAQMTQAAFgAoBHhOAAAYBAxNAAAREQAAANEB8NADAAAAjgMBCDMAAQID
BAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJQkmJygpKissGC0uLzAxMjM0NS82CTM3ODk6Gzs0PD0+P0BBQkNERTBB
RkcnSElKSwZMTU4FT1A4UTpSU1RVSRFWV1hZWltcXV5fYGFiY2RlZmdoOwZpQGoYa2xtbmlvVXBxcnN0GHV2d3h5Anp7JnxifX5/
gIFhggSDMkBfLoSFN4YSZTWHiImKi4wsjY6PIJCRSDiSk5SVlpeYmZpXQ4+bABUEFZ4PFagPTBWGAxUAEgAAzwf0zgMAAA5wGAAA
FeAAAAAM8hAAAAHPrAAAGCIgAAAddDwAAAyZAAAADae0AAAPiPQAABcwuAAAGPcEAAAOpTgAABzwmAAAHgz4AAAB4/wAAAYu0AAA
G2CYAAAD5UQAABGEYAAAEWvEAAANt1QAABw+3AAAF9M4AAAI8KwAABzyjAAAFpAsAAAXR2AAAASM2AAACswQAAAD9BwAAAC7gAAA
F/1oAAAWlaQAAA+AwAAAGNVsAAAYNnAAAAr3CAAAE5uIAAAPPSwAABNpiAAAHWoUAAAaA8QAAAWilAAAECbEAAAbVnAAAACKSAAA
AqzEAAAZsbwAAAY6JAAADNz4AAAcL6AAABd4mAAABv8EAAAPSnQAABxmtAAADGroAAAcp4wAAAlsgAAADosMAAAaWnwAAAKCMAAA
ECLcAAAQKkgAABFGqAAADrR0AAASI8AAABANxAAAFdcEAAAFYCwAABOnpAAAFbaYAAAbwwgAAAPhwAAAC6q4AAAbAtgAABE3CAAA
HJsMAAAdo4AAAAsnFAAADUR8AAAbe/AAAAPnOAAAEjI0AAAa5xwAAAfonAAAGymEAAAF3GQAABTV/AAAAov0AAAeQ0QAABBGBAAA
GgbkAAAdfAwAAApfFAAABgxwAAAQkWgAAA5i0AAAGMoYAAAPdKQAABSCAAAAG53sAAAHX+QAABqK7AAABBjUAAAdqiQAAAzRQAAA
D3iMAAACeAgAABntHAAAGLzQAAAQtPQAAAiIxAAADEYwAAABrUwAABw+eAAADC34AAAMp3QAABBw/AAADkS8AAAb+PAAAAyl5AAA
HB4MAAAbXRQAAAmMJAAAG9lMAAAV4rwAABjeBAAABI4EAAANX3AAABttGAAAG0QUAAAVm0AAABNnlAAAE2kkAAAEFVAAAADAMAAA
FuL8AAAF4EwAABGZeAAACSgkAAACc1gAABb+uAAAFYiAAAAeLpAAAA5pdAAAHaj4AAAKHKwAABqFEAAAEuqUAAAKbFwAAA0C3AAA
E/SYAAAIEzAAAB1I4AAAAKIcAAAUPtAAAAzujAAAFQq4AAASkLwAAAvVsAAAF7AQAAAKDEQAAApOrAAAFj9QAAAdPSgAABTgJAAA
HT2MAAAT5iQAABkrwAAAAHngAAAJipQAAAaCBAAAEb1oAAAJTtAAABY7BAAADDMMAAAH3BwAAAY4+AAADDdYAAAJoTwAABxTkAAA
Cw+kAAAXsNgAABM4tAAAGDCUAAAXcSwAABF5DAAAFKvMAAAEVDQAAAuwMAAACC7sAAAZ2MwAABWiSAAAF8B4AAAKCSQVABWiAxWq
AywVjgMVEBUGFQYcGAUAAB5DRBgFAAAAeeAWACgFAAAeQ0QYBQAAAHngEREAAADRAfDQAwAAAI4DAQgzAAECAwQFBgcICQoLDA0O
DxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OSc6Ozw9Pj9AQUJDREVGR0hJSktMTU4UT1BRUlNUVVZX
WFlWWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6Ch
oqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8vYS+v8DBwgAVBBlcNQAYBnNjaGVtYRUIABUEJQIYCG9yZGVyX2lkABUEJQIYC2N1
c3RvbWVyX2lkABUCJQIYCm9yZGVyX2RhdGUlDExsAAAAFQ4VChUCGAZhbW91bnQlChUEFRQsXBUEFRQAAAAWjgMZHBlMJgAcFQQZ
NQAGEBkYCG9yZGVyX2lkFQIWjgMWuB0WphEmgA0mCBwYCEc5AQAAAAAAGAiBOAEAAAAAABYAKAhHOQEAAAAAABgIgTgBAAAAAAAR
EQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAjgMAAAAmABwVBBk1AAYQGRgLY3VzdG9tZXJfaWQVAhaOAxaABxa+BSbUEyauERwYCB8D
AAAAAAAAGAi9AgAAAAAAABYAKAgfAwAAAAAAABgIvQIAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAjgMAAAAmABwVAhk1
AAYQGRgKb3JkZXJfZGF0ZRUCFo4DFogOFpoOJvggJuwWHBgEeE4AABgEDE0AABYAKAR4TgAAGAQMTQAAEREAGSwVBBUAFQIAFQAV
EBUCADwpBhkmAI4DAAAAJgAcFQ4ZNQAGEBkYBmFtb3VudBUCFo4DFs4TFuATJtA0JoYlHBgFAAAeQ0QYBQAAAHngFgAoBQAAHkNE
GAUAAAB54BERABksFQQVABUCABUAFRAVAgA8KQYZJgCOAwAAABaORhaOAyYIFt44ABkcGAxBUlJPVzpzY2hlbWEYoAMvLy8vL3pB
QkFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBUUFBQURFQUFBQWZB
QUFBRUFBQUFBRUFBQUFYUC8vL3dBQUFRY1FBQUFBSUFBQUFBUUFBQUFBQUFBQUJnQUFBR0Z0YjNWdWRBQUFDQUFNQUFRQUNBQUlB
QUFBQ2dBQUFBSUFBQUNVLy8vL0FBQUJDQkFBQUFBa0FBQUFCQUFBQUFBQUFBQUtBQUFBYjNKa1pYSmZaR0YwWlFBQUFBQUdBQWdB
QmdBR0FBQUFBQUFBQU16Ly8vOEFBQUVDRUFBQUFCd0FBQUFFQUFBQUFBQUFBQXNBQUFCamRYTjBiMjFsY2w5cFpBQzgvLy8vQUFB
QUFVQUFBQUFRQUJRQUNBQUdBQWNBREFBQUFCQUFFQUFBQUFBQUFRSVFBQUFBSkFBQUFBUUFBQUFBQUFBQUNBQUFBRzl5WkdWeVgy
bGtBQUFBQUFnQURBQUlBQWNBQ0FBQUFBQUFBQUZBQUFBQQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGUwcAAAc
AAAcAAAcAAAAAAQAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "019-scd2-point-in-time-lookup"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

customer_dim = spark.read.parquet((_data_dir / "customer_dim.parquet").resolve().as_posix())
customer_dim.createOrReplaceTempView("customer_dim")
orders = spark.read.parquet((_data_dir / "orders.parquet").resolve().as_posix())
orders.createOrReplaceTempView("orders")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'amount': 'ba5ee94a4579fbcd8f2de6920ccffe94f464a872509f6000c10fbc0573935421',
                         'customer_id': 'a23dc91d614c6e6544ef0b81fb0e9f6ba1203a0bf9f2b21e82b9051b4d267bf0',
                         'order_date': '8804f7ffa4503bcc57fbfda42f30021f467d7a6cdd13a53d2efaa000a2d2b5a5',
                         'order_id': 'e4c3a5d0b6b8aac92afafa703866f9f786167505a715bb9d4e7e83dcf24fa3cd',
                         'tier': 'f836d0d43ad5bb06e8a74827a6d4bd33bc73cdd1aa3ad46d9e68b51984603a54'},
 'columns': [['amount', 'decimal(10,2)'],
             ['customer_id', 'bigint'],
             ['order_date', 'date'],
             ['order_id', 'bigint'],
             ['tier', 'string']],
 'fingerprint': '4a7e9c5ee035a9a82a51c0a461953baced395b85489e2359dea25bbf5894f2c8',
 'order_matters': False,
 'row_count': 199}

check = make_check(EXPECTED)

## The situation

Customers move between loyalty tiers over time. `customer_dim` keeps the history as a
type 2 slowly changing dimension: one row per period during which a customer stayed in
one tier.

Marketing wants to know how much each tier spends. For that, every order has to carry
the tier the customer was in on the day of the order, not the tier they are in today.

## Input tables

### `customer_dim`

One row per tier period of a customer.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | The customer. |
| `tier` | STRING | `SILVER`, `GOLD` or `PLATINUM`. |
| `valid_from` | DATE | First day of the period. |
| `valid_to` | DATE | The day the period ended. The customer was no longer in this tier on that day. NULL for the customer's current period. |

A customer's periods never overlap. Usually the next period starts on the day the
previous one ends, but a customer can also have been out of the programme for a while.

### `orders`

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | Unique per order. |
| `customer_id` | BIGINT | The customer who ordered. Not every customer is in `customer_dim`. |
| `order_date` | DATE | Date of the order. |
| `amount` | DECIMAL(10,2) | Order value in rupees. |

## Requirement

Return every order exactly once, with the customer's tier on the order date.

- A period covers `valid_from` and every later day **before** `valid_to`. An order
  placed on the day a customer changes tier belongs to the new tier.
- A period with a NULL `valid_to` covers every day from `valid_from` onwards.
- If no period of the customer covers the order date, `tier` is NULL. That happens
  for orders before the customer's first period, during a break in the programme,
  and for customers that are not in `customer_dim`.

## Expected output

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | |
| `customer_id` | BIGINT | |
| `order_date` | DATE | |
| `amount` | DECIMAL(10,2) | |
| `tier` | STRING | The tier on the order date. NULL if no period covers it. |

In [ ]:
customer_dim.show(5)
orders.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `customer_dim`, `orders`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)